# Step 4: Beat & Downbeat Tracking with `module.beat_tracking` & `beat_this`

This notebook uses **`module.beat_tracking`** to perform automatic beat and downbeat tracking on audio using the **`beat_this`** state-of-the-art transformer model.

### Pipeline:
1. **Select Input Audio**: Analyzes `data/music.mp3` (or `data/drums.wav` / `data/instrumental.wav`).
2. **Initialize BeatThis Model**: Uses `get_beat_model()` on GPU (`cuda`).
3. **Predict Beats & Downbeats**: Runs transformer inference with `predict_beats(..., show_progress=True)`.
4. **Export CSV**: Saves beat timestamps and meter numbers to **`data/beats.csv`**.
5. **Rhythm & Tempo Analysis**: Displays estimated BPM, downbeat distribution, and bar structure.

> **Kernel**: Make sure the kernel is set to **`Python (seperate)`**.

## 1. Setup & Device Configuration

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path and set as current working directory
PROJECT_ROOT = Path("..").resolve() if Path("..", "module").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from pathlib import Path
import torch
import numpy as np
import pandas as pd
from module.beat_tracking import get_beat_model, predict_beats, save_beats

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Define Input & Output Paths

In [ ]:
data_dir = Path("data")
# Track beats on full mix (music.mp3) or isolated stems
music_path = data_dir / "music.mp3"
drums_path = data_dir / "drums.wav"

input_audio = music_path if music_path.exists() else drums_path
beats_csv_path = data_dir / "beats.csv"

print(f"Selected Input Audio : {input_audio}")
print(f"Beats CSV Target     : {beats_csv_path}")

assert input_audio.exists(), f"Missing audio file at: {input_audio}. Please place audio in data/ first!"
print("\n✓ Input audio file found!")

## 3. Load BeatThis Model via `module.beat_tracking`

In [ ]:
print(f"Loading BeatThis model on {device}...")
beat_model = get_beat_model(device=device)
print("✅ BeatThis model ready for inference!")

## 4. Run Beat & Downbeat Tracking

In [ ]:
# Run inference
results = predict_beats(input_audio, model=beat_model, show_progress=True)

df_beats = results["df"]
print(f"\nExtracted {len(df_beats)} beats across the track.")
print(f"Estimated Tempo: {results['bpm']} BPM")
print("\nFirst 15 detected beats:")
print(df_beats.head(15).to_string(index=False))

## 5. Save Beat Timestamps (CSV)

In [ ]:
save_beats(results, beats_csv_path)
print(f"✅ Successfully saved beats to CSV: {beats_csv_path}")

## 6. Rhythm, Tempo & Measure Overview

In [ ]:
total_beats = len(results["beats"])
total_downbeats = len(results["downbeats"])

print("📊 Beat Tracking Statistics:")
print(f"  • Total Beats      : {total_beats}")
print(f"  • Total Downbeats  : {total_downbeats} measures / bars")
print(f"  • Estimated Tempo  : {results['bpm']} BPM")

if total_beats > 1:
    intervals = np.diff(results["beats"])
    print(f"  • Mean Beat Interval: {np.mean(intervals):.3f} seconds")
    print(f"  • Min / Max Interval: {np.min(intervals):.3f}s / {np.max(intervals):.3f}s")

# Show measure breakdown preview
print("\n🎵 First 6 Bars / Measures Preview:")
measure_num = 0
measure_beats = []
for row in df_beats.itertuples():
    if row.is_downbeat:
        if measure_beats:
            print(f"  Measure {measure_num:>2}: [" + ", ".join(f"{t:.2f}s (b{b})" for t, b in measure_beats) + "]")
            measure_num += 1
            if measure_num > 6:
                break
            measure_beats = []
        if measure_num == 0:
            measure_num = 1
    measure_beats.append((row.time, row.beat))

if measure_beats and measure_num <= 6:
    print(f"  Measure {measure_num:>2}: [" + ", ".join(f"{t:.2f}s (b{b})" for t, b in measure_beats) + "]")